# Entraînement de l'encodeur (pas-à-pas)

Ce notebook reprend les scripts `src/build_encoder_model.py` et `src/train_encoder_model.py` pour exécuter l'entraînement étape par étape.

## 1) Imports et accès au projet

In [1]:
# Imports standard et TensorFlow.
from pathlib import Path
import sys
import tensorflow as tf

# Détecte la racine du projet (dossier qui contient src).
project_root = Path.cwd()
if (project_root / 'src').exists():
    pass
elif (project_root.parent / 'src').exists():
    project_root = project_root.parent
else:
    raise RuntimeError('Impossible de trouver le dossier src depuis le notebook.')

# Ajoute la racine du projet au PYTHONPATH pour importer src/.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)


Project root: /Users/lewagon/code/Lionel-Nokia/lab/art-xplain/art-xplain


## 2) Charger la config et les fonctions

In [2]:
# Importe les utilitaires et la fonction de build de l'encodeur.
from src.utils import load_config, ensure_dir
from src.build_encoder_model import build_style_encoder_model

# Charge la config YAML du projet.
cfg = load_config(project_root / 'config/config.yaml')
# Chemins utiles pour le dataset et les modèles.
keras_root = project_root / cfg['paths']['keras_root']
models_root = project_root / cfg['paths']['models_root']

print('keras_root:', keras_root)
print('models_root:', models_root)


keras_root: /Users/lewagon/code/Lionel-Nokia/lab/art-xplain/art-xplain/data/out
models_root: /Users/lewagon/code/Lionel-Nokia/lab/art-xplain/art-xplain/models


## 3) Vérifier les datasets train/val

In [3]:
# Les dossiers train/val doivent exister avant l'entraînement.
train_dir = keras_root / 'train'
val_dir = keras_root / 'val'

print('train_dir exists:', train_dir.exists())
print('val_dir exists:', val_dir.exists())
if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError("Dossiers train/val introuvables. Lance d'abord build_dataset_from_csv.")


train_dir exists: True
val_dir exists: True


## 4) Charger les hyperparamètres de config

In [ ]:
# Hyperparamètres du modèle (taille image, embedding, backbone).
img_size = int(cfg['model']['img_size'])
embed_dim = int(cfg['model']['embed_dim'])
backbone = str(cfg['model']['backbone'])
freeze_backbone = bool(cfg['model'].get('freeze_backbone', True))

# Hyperparamètres d'entraînement.
train_cfg = cfg.get('train', {})
batch_size = int(train_cfg.get('batch_size', 32))
epochs_head = int(train_cfg.get('epochs_head', 8))
epochs_finetune = int(train_cfg.get('epochs_finetune', 10))
lr_head = float(train_cfg.get('lr_head', 1e-3))
lr_finetune = float(train_cfg.get('lr_finetune', 1e-5))
finetune_last_layers = int(train_cfg.get('finetune_last_layers', 60))
early_stopping_patience = int(train_cfg.get('early_stopping_patience', 4))

# Paramètres dataset.
data_cfg = cfg.get('dataset', {})
max_images = int(data_cfg.get('max_images', 5000))
keep_top_styles = int(data_cfg.get('keep_top_styles', 5))
min_images_per_style = int(data_cfg.get('min_images_per_style', 200))

print("-- Paramètres dataset --")
print('max_images:', max_images)
print('keep_top_styles:', keep_top_styles)
print('min_images_per_style:', min_images_per_style)


print("\n-- Hyperparamètres d'entraînement --")
print('img_size:', img_size)
print('embed_dim:', embed_dim)
print('backbone:', backbone)
print('freeze_backbone:', freeze_backbone)
print('batch_size:', batch_size)
print('epochs_head:', epochs_head, 'epochs_finetune:', epochs_finetune)


-- Paramètres dataset --
max_images: 6500
keep_top_styles: 5

-- Hyperparamètres d'entraînement --
img_size: 224
embed_dim: 256
backbone: EfficientNetV2B0
freeze_backbone: True
batch_size: 32
epochs_head: 8 epochs_finetune: 10


In [6]:
data_cfg = cfg.get('dataset', {})
max_images = int(data_cfg.get('max_images', 5000))

print('max_images:', max_images)


max_images: 6500


## 5) Construire les datasets TensorFlow

In [5]:
# Charge les images depuis les dossiers et crée les batches.
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode='int',
    image_size=(img_size, img_size),
    batch_size=batch_size,
    shuffle=True,
)

# Validation: même classes et pas de shuffle.
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    label_mode='int',
    image_size=(img_size, img_size),
    batch_size=batch_size,
    shuffle=False,
    class_names=train_ds.class_names,
)

class_names = train_ds.class_names
n_classes = len(class_names)

print('classes:', class_names)
print('n_classes:', n_classes)

# Optimisation du pipeline d'entrée.
# optimisation standard du pipeline tf.data pour ne pas bloquer le GPU/CPU pendant l’entraînement.
# tf.data.AUTOTUNE laisse TensorFlow choisir automatiquement le niveau de pré‑lecture.
# prefetch() prépare les prochains batches pendant que le modèle entraîne sur le batch courant.
# Effet:
# Moins d’attente I/O.
# Meilleure utilisation CPU/GPU.
# Pipeline plus fluide sans changer les données.

autotune = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(autotune)
val_ds = val_ds.prefetch(autotune)


Found 39840 files belonging to 10 classes.
Found 4991 files belonging to 10 classes.
classes: ['Abstract_Expressionism', 'Art_Nouveau_Modern', 'Baroque', 'Expressionism', 'Impressionism', 'Northern_Renaissance', 'Post_Impressionism', 'Realism', 'Romanticism', 'Symbolism']
n_classes: 10


## 6) Construire l'encodeur (backbone + embedding)

In [ ]:
# Construit l'encodeur (backbone + embedding L2).
encoder = build_style_encoder_model(
    img_size=img_size,
    embed_dim=embed_dim,
    backbone=backbone,
    freeze_backbone=freeze_backbone,
)
encoder.summary()


## 7) Construire le classifieur (tête softmax)

In [ ]:
# Ajoute une tête softmax pour la classification pendant l'entraînement.
inputs = tf.keras.Input(shape=(img_size, img_size, 3), name='image')
x = encoder(inputs)
logits = tf.keras.layers.Dense(n_classes, activation='softmax', name='style_head')(x)
classifier = tf.keras.Model(inputs, logits, name='styledna_classifier')
classifier.summary()


## 8) Callbacks et phase 1 (tête)

In [ ]:
# Callbacks: early stopping et réduction du LR en plateau.
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=early_stopping_patience,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-7,
    ),
]

# Phase 1: entraînement de la tête (backbone gelé).
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_head),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

print(f'Phase 1/2: entraînement de la tête ({epochs_head} epochs)')
history_head = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_head,
    callbacks=callbacks,
)


## 9) Phase 2 (fine-tuning optionnel)

In [ ]:
# Fonctions utilitaires pour le fine-tuning.
def _get_backbone_from_encoder(encoder_model: tf.keras.Model) -> tf.keras.Model:
    for layer in reversed(encoder_model.layers):
        if isinstance(layer, tf.keras.Model):
            return layer
    raise ValueError("Backbone introuvable dans l'encodeur.")

def _set_finetune_layers(backbone_model: tf.keras.Model, n_last_layers: int) -> None:
    backbone_model.trainable = True
    if n_last_layers <= 0:
        for layer in backbone_model.layers:
            layer.trainable = False
        return
    split_idx = max(0, len(backbone_model.layers) - n_last_layers)
    for i, layer in enumerate(backbone_model.layers):
        layer.trainable = i >= split_idx

# Phase 2: fine-tuning optionnel.
if epochs_finetune > 0:
    backbone_model = _get_backbone_from_encoder(encoder)
    _set_finetune_layers(backbone_model, finetune_last_layers)

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_finetune),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )

    print(
        f'Phase 2/2: fine-tuning ({epochs_finetune} epochs, '
        f'{finetune_last_layers} dernières couches)'
    )
    history_ft = classifier.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs_finetune,
        callbacks=callbacks,
    )
else:
    print('Fine-tuning ignoré (epochs_finetune=0)')


## 10) Sauvegarder l'encodeur

In [ ]:
# Sauvegarde l'encodeur entraîné pour les étapes embeddings/retrieval.
ensure_dir(models_root)
encoder.save(models_root / 'encoder.keras')
print('Saved encoder:', (models_root / 'encoder.keras').resolve())
print('Classes:', class_names)


# Model building

### Étapes du modèle (encodeur + entraînement)

#### - Entrée image
- Une image est chargée et redimensionnée en img_size × img_size × 3.

#### - Prétraitement EfficientNetV2

- Normalisation/scale adaptée au backbone EfficientNetV2.

#### - Backbone EfficientNetV2B0

- Réseau convolutionnel pré-entraîné ImageNet (sans la tête finale).

#### - GlobalAveragePooling2D

- Agrège les cartes de features en un vecteur fixe.

#### - Dense (projection)

- Projection vers la dimension d’embedding embed_dim (ex: 256).

#### - UnitNormalization (L2)

- Normalise l’embedding sur la sphère unitaire pour la similarité cosinus.

#### - (Entraînement uniquement) Tête de classification

- Dense + softmax vers n_classes styles.
- Deux phases:
  - Phase 1: entraînement de la tête (backbone gelé).
  - Phase 2 (optionnel): fine‑tuning des dernières couches du backbone.

#### - Usage

- Retrieval: on garde l’embedding L2 pour comparer les images.
- Grad‑CAM: visualisation des zones qui expliquent la similarité.

### GlobalAveragePooling2D

- **GlobalAveragePooling2D** prend un tenseur de forme (H, W, C) et moyenne chaque carte de caractéristiques sur les dimensions spatiales (H, W).

- Résultat:

  - Entrée: H × W × C
  -Sortie: C

- Intuition:

    - Chaque carte (canal) devient un seul nombre, représentant sa présence moyenne dans l’image.
    - Ça réduit fortement la dimension et évite une couche dense très lourde.

    - Exemple simple:

      - Si la sortie du backbone est (7, 7, 1280), le pooling produit un vecteur (1280,).

- Pourquoi c’est utilisé:

  - Réduction de dimension simple et stable.
  - Moins de paramètres que Flatten + Dense.
  - Bon comportement pour la classification et l’embedding.